## Limpieza de datos, archivo: Orders

In [2]:
import pandas as pd
RAW = "../../data/raw/"

# Cargas con parse_dates donde corresponde
orders = pd.read_csv(
    RAW + "olist_orders_dataset.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ]
)

In [ ]:
# 1) Define qué estados requieren cada fecha
must_have_approved = {"shipped", "delivered", "invoiced", "processing"}
must_have_carrier  = {"shipped", "delivered"}
must_have_customer = {"delivered"}

cond_ok = pd.Series(True, index=orders.index)

# Si el estado exige approved_at, debe existir
cond_ok &= ~(
    orders["order_status"].isin(must_have_approved)
    & orders["order_approved_at"].isna()
)

# Si el estado exige delivered_carrier_date, debe existir
cond_ok &= ~(
    orders["order_status"].isin(must_have_carrier)
    & orders["order_delivered_carrier_date"].isna()
)

# Si el estado exige delivered_customer_date, debe existir
cond_ok &= ~(
    orders["order_status"].isin(must_have_customer)
    & orders["order_delivered_customer_date"].isna()
)

# 2) Mantén todo lo consistente; separa lo inconsistente para auditoría
orders_clean = orders[cond_ok].copy()


In [7]:
print(f"Filas originales:   {len(orders):,}")
print(f"Filas consistentes: {len(orders_clean):,}")

Filas originales:   99,441
Filas consistentes: 96,461


In [8]:
# Verificar valores nulos y duplicados del DataFrame orders
print("=" * 70)
print("VALORES NULOS Y DUPLICADOS DEL DATAFRAME: orders")
print("=" * 70)

# Valores nulos por columna
null_values_orders = orders_clean.isnull().sum()

# Verificar duplicados
duplicates_orders = orders_clean.duplicated().sum()

# Mostrar resultados
print("\nValores nulos por columna:\n", null_values_orders)
print("\nNúmero de registros duplicados:", duplicates_orders)
print("\nTotal de registros:", len(orders_clean))
print("Porcentaje de duplicados: {:.2f}%".format((duplicates_orders / len(orders_clean)) * 100))

VALORES NULOS Y DUPLICADOS DEL DATAFRAME: orders

Valores nulos por columna:
 order_id                         0
customer_id                      0
order_status                     0
order_purchase_timestamp         0
order_approved_at                0
order_delivered_carrier_date     0
order_delivered_customer_date    0
order_estimated_delivery_date    0
dtype: int64

Número de registros duplicados: 0

Total de registros: 96461
Porcentaje de duplicados: 0.00%
